# SmartQ — Data Preparation

This notebook prepares the SmartQ dataset for waiting-time regression after EDA.

The target is `actual_wait_minutes`. Only completed visits are used for regression.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

DATASET_NAME = 'SmartQ_Synthetic_Operational_Dataset_100k.csv'
candidate_paths = [Path('data') / DATASET_NAME, Path('..') / 'data' / DATASET_NAME]
data_path = next((p for p in candidate_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError('SmartQ dataset not found.')

df = pd.read_csv(data_path, parse_dates=['scenario_date'])
completed = df[df['status'] == 'COMPLETED'].copy()
print(f'Completed visits: {len(completed):,}')


## 1. Final modelling features

Only information available at prediction/check-in time is used. Outcome fields and the existing `baseline_eta_minutes` are excluded from the official model inputs.


In [ ]:
TARGET = 'actual_wait_minutes'

numeric_features = [
    'arrival_offset_minutes', 'people_ahead', 'general_waiting', 'priority_waiting',
    'serving_count', 'open_general_counters', 'open_priority_counters',
    'effective_open_counters', 'counter_utilisation', 'queue_pressure_index',
    'workload_minutes_ahead', 'recent_avg_service_minutes_10',
    'recent_avg_wait_minutes_10', 'recent_throughput_60m',
    'service_target_minutes', 'hour_of_day'
]

categorical_features = [
    'branch_code', 'service_code', 'booking_source',
    'queue_type', 'day_of_week', 'is_peak_period'
]

features = numeric_features + categorical_features
print(f'Input features: {len(features)}')


## 2. Missing-value strategy

The two recent-history features can be missing early in an operating day. Numeric missing values are imputed with the **training-set median**. Categorical missing values use the most frequent training category. This prevents information from validation/test data leaking into preprocessing.


## 3. Chronological train / validation / test split

The split is done by **whole operating dates**, so one day never appears in more than one partition.

- Training: 2026-01-02 to 2026-06-16
- Validation: 2026-06-17 to 2026-07-22
- Test: 2026-07-23 to 2026-08-27


In [ ]:
dates = sorted(completed['scenario_date'].dt.normalize().unique())
n_dates = len(dates)

train_end = pd.Timestamp(dates[int(n_dates * 0.70) - 1])
val_start = pd.Timestamp(dates[int(n_dates * 0.70)])
val_end = pd.Timestamp(dates[int(n_dates * 0.85) - 1])
test_start = pd.Timestamp(dates[int(n_dates * 0.85)])

train = completed[completed['scenario_date'] <= train_end].copy()
validation = completed[(completed['scenario_date'] >= val_start) & (completed['scenario_date'] <= val_end)].copy()
test = completed[completed['scenario_date'] >= test_start].copy()

split_summary = pd.DataFrame({
    'split': ['Train', 'Validation', 'Test'],
    'rows': [len(train), len(validation), len(test)],
    'start': [train['scenario_date'].min().date(), validation['scenario_date'].min().date(), test['scenario_date'].min().date()],
    'end': [train['scenario_date'].max().date(), validation['scenario_date'].max().date(), test['scenario_date'].max().date()],
})
display(split_summary)


### Confirmed split sizes

- **Train:** 64,074 rows (69.15%)
- **Validation:** 14,665 rows (15.83%)
- **Test:** 13,916 rows (15.02%)

Total regression population: **92,655 completed visits**.


In [ ]:
X_train, y_train = train[features], train[TARGET]
X_validation, y_validation = validation[features], validation[TARGET]
X_test, y_test = test[features], test[TARGET]

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features),
])

X_train_prepared = preprocessor.fit_transform(X_train)
X_validation_prepared = preprocessor.transform(X_validation)
X_test_prepared = preprocessor.transform(X_test)

print('Prepared shapes:')
print('Train:', X_train_prepared.shape)
print('Validation:', X_validation_prepared.shape)
print('Test:', X_test_prepared.shape)


## 4. Leakage protection

These fields are deliberately excluded from model inputs: `call_time`, `actual_wait_minutes`, `wait_variance_minutes`, `actual_service_minutes`, `service_variance_minutes`, `counter_number`, `service_started_at`, `service_completed_at`, status-derived outcome flags, and the existing deterministic `baseline_eta_minutes`.

The prepared data is now ready for the mean baseline, Linear Regression, Random Forest and XGBoost.
